# Advanced Machine Learning Final Project
- Author: César Núñez
- Date: March 10th 2026

## Data preparation

In [99]:
import json, random, re, nltk
from nltk.corpus import stopwords
from pathlib import Path
import polars as pl
from social_conflicts_peru.config import directories
from string import punctuation
from nltk.tokenize import word_tokenize
from loguru import logger
import torch
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, balanced_accuracy_score

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding


In [100]:
with open(directories.PROCESSED_DATA / "conflict_occurrences_active_latent.json" , "r") as file:
    data = pl.DataFrame(json.load(file))

In [101]:
data.columns = ['conflict_type', 'date_init', 'conflict_text', 'location', 'actors_1', 'actors_2', 'actors_3', 'event_text', 
                'dialogo', '_page_start', '_page_end', '_confidence', '_section', '_subsection', '_source_pdf', '_seccion', 
                '_grupo_conflicto', '_case_id_local', 'conflict_uid', 'state', 'report_number']

In [102]:
# Creating variables
data = data.filter(pl.col("dialogo").is_in(["HAY DIÁLOGO", "NO HAY DIÁLOGO"]))
data = data.filter(pl.len().over("conflict_uid") > 5).sort(["conflict_uid", "report_number"])

In [103]:
# Cleaning types
"Asunto de gobierno regional"
"Asuntos de gobierno regional"

data = data.with_columns(
    pl.when(
        pl.col("conflict_type").str.contains(
            r"(?i)^por asuntos? de gobierno local|^asuntos? de gobierno local"
        )
    )
    .then(pl.lit("Asuntos de gobierno local"))

    .when(
        pl.col("conflict_type").str.contains(
            r"(?i)^por asuntos? de gobierno regional|^asuntos? de gobiern(os)? regional|^asuntos? de gobiernos regional|^asuntos? de gobierno regional"
        )
    )
    .then(pl.lit("Asuntos de gobierno regional"))

    .when(
        pl.col("conflict_type").str.contains(
            r"(?i)^asuntos? de gobierno nacional"
        )
    )
    .then(pl.lit("Asuntos de gobierno nacional"))

    .when(
        pl.col("conflict_type").str.contains(
            r"(?i)^cultivo ilegal de hoja de coca"
        )
    )
    .then(pl.lit("Cultivo ilegal de hoja de coca"))

    .when(
        pl.col("conflict_type").str.contains(
            r"(?i)^demarcación territorial"
        )
    )
    .then(pl.lit("Demarcación territorial"))

    .when(
        pl.col("conflict_type").str.contains(
            r"(?i)^socioambiental"
        )
    )
    .then(pl.lit("Socioambiental"))

    .otherwise(pl.col("conflict_type"))
    .alias("conflict_type")
)

In [104]:
month_map = {
    "enero": "01",
    "febrero": "02",
    "marzo": "03",
    "abril": "04",
    "mayo": "05",
    "junio": "06",
    "julio": "07",
    "agosto": "08",
    "septiembre": "09",
    "setiembre": "09",
    "octubre": "10",
    "noviembre": "11",
    "diciembre": "12",
}

month_pattern = "|".join(month_map.keys())

data = (
    data
    .with_columns(
        pl.col("date_init").str.strip_chars().alias("date_init")
    )
    .with_columns(
        pl.col("date_init")
        .str.extract(rf"(?i)^({month_pattern})", 1)
        .str.to_lowercase()
        .replace(month_map)
        .alias("month_num"),

        pl.col("date_init")
        .str.extract(r"(?i)(\d{4})", 1)
        .alias("year"),

        pl.col("date_init")
        .str.extract(
            rf"(?i)^(?:{month_pattern})(?:\s+de)?[,\s]+(?:\d{{4}})\.?,?\s*(.*)$",
            1
        )
        .alias("rest_text"),
    )
    .with_columns(
        pl.when(
            pl.col("year").is_not_null() & pl.col("month_num").is_not_null()
        )
        .then(
            pl.concat_str([
                pl.col("year"),
                pl.lit("-"),
                pl.col("month_num"),
                pl.lit("-01"),
            ]).str.strptime(pl.Date, "%Y-%m-%d")
        )
        .otherwise(None)
        .alias("clean_date"),

        pl.when(pl.col("rest_text") == "")
        .then(None)
        .otherwise(pl.col("rest_text"))
        .alias("rest_text"),
    )
    .drop(["month_num", "year"])
)

In [105]:
data = data.with_columns(
    pl.col("event_text").fill_null(pl.col("rest_text")).alias("event_text")
)

In [106]:
data = data.filter(
    pl.col('event_text') != 'se registraron nuevos hechos durante el mes.',
    pl.col('event_text') != 'se registraron nuevos hechos durante el mes..',
    pl.col('conflict_type') != 'Cultivo ilegal de hoja de coca' ## There are only 2 examples
)

In [107]:
data = data.with_columns(
    pl.col('conflict_type').str.to_lowercase().alias('conflict_type')
)

In [108]:
df_final = data[:, ["conflict_type", "date_init", "conflict_text", "location", "actors_1", "actors_2", "actors_3", "event_text", "dialogo", "conflict_uid", "report_number", "clean_date"]]

In [109]:
df_final["conflict_type"].value_counts()

conflict_type,count
str,u32
"""otros asuntos""",35
"""comunal""",206
"""asuntos de gobierno nacional""",300
"""asuntos de gobierno regional""",171
"""socioambiental""",2029
"""asuntos de gobierno local""",93
"""laboral""",61
"""demarcación territorial""",52


In [110]:
df_final["dialogo"].value_counts()

dialogo,count
str,u32
"""NO HAY DIÁLOGO""",761
"""HAY DIÁLOGO""",2186


### Feature extraction

In [111]:
# Dialogo to dummy variable
df_final = df_final.with_columns(
    pl.col("dialogo")
    .replace({
        "HAY DIÁLOGO": 1,
        "NO HAY DIÁLOGO": 0
    })
    .cast(pl.Int8)
    .alias("dialogo")
)

In [112]:
df_final = df_final.unique(
    subset=["conflict_uid", "report_number", "event_text"],
    keep="first"
)

### Class imbalance

In [113]:
pl.DataFrame({
    "total_rows": [df_final.height],
    "positive": [df_final.filter(pl.col("dialogo") == 1).height],
    "negative": [df_final.filter(pl.col("dialogo") == 0).height],
}).with_columns(
    (pl.col("positive") / pl.col("total_rows")).alias("positive_ratio"),
    (pl.col("negative") / pl.col("total_rows")).alias("negative_ratio")
)

total_rows,positive,negative,positive_ratio,negative_ratio
i64,i64,i64,f64,f64
2945,2184,761,0.741596,0.258404


### Text cleaning

In [114]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
SPA_STOP = stopwords.words("spanish")

def clean_text(text: str, stopwords: list[str] = SPA_STOP) -> str:

    words = word_tokenize(text.lower())
    clean_words = [word for word in words if word not in stopwords]
    return " ".join(clean_words)

[nltk_data] Downloading package stopwords to /home/canun/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/canun/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/canun/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [115]:
df_final = df_final.with_columns(
    pl.col('conflict_text')
    .map_elements(clean_text, return_dtype=pl.String)
    .alias('clean_conflict_text'),
    pl.col('event_text')
    .map_elements(clean_text, return_dtype=pl.String)
    .alias('clean_event_text')
)

In [116]:
df_final = df_final[:, ["dialogo", "conflict_type", "conflict_uid", "report_number", "clean_date", "clean_conflict_text", "clean_event_text", "date_init", "location", "actors_1", "actors_2", "actors_3"]]

## Classification

This section adds two independent classifiers:

1. `conflict_type` multiclass model using conflict-level rows (`clean_conflict_text`).
2. `dialogo` binary model using event-level rows (`clean_event_text`) with group split by `conflict_uid`.

Both pipelines use consistent label mapping, train/val/test separation, and macro-F1 model selection.

In [33]:
# Shared utilities for robust splits + transformer training
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import polars as pl
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    average_precision_score,
)
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer,
)

In [45]:
SEED = 42

@dataclass
class LabelMapping:
    label2id: Dict[str, int]
    id2label: Dict[int, str]


def build_label_mapping(values: list[str]) -> LabelMapping:
    labels = sorted(set(values))
    label2id = {label: ix for ix, label in enumerate(labels)}
    id2label = {ix: label for label, ix in label2id.items()}
    return LabelMapping(label2id=label2id, id2label=id2label)


def compute_metrics_factory(num_labels: int):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
            labels, preds, average="weighted", zero_division=0
        )

        metrics = {
            "accuracy": accuracy_score(labels, preds),
            "balanced_accuracy": balanced_accuracy_score(labels, preds),
            "precision_weighted": precision_w,
            "recall_weighted": recall_w,
            "f1_weighted": f1_w,
            "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        }

        if num_labels == 2:
            probs_pos = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
            metrics["pr_auc_positive"] = average_precision_score(labels, probs_pos)

        return metrics

    return compute_metrics


class WeightedTrainer(Trainer):
    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


def train_text_classifier(
    train_df: pl.DataFrame,
    val_df: pl.DataFrame,
    test_df: pl.DataFrame,
    text_col: str,
    label_col: str,
    model_name: str,
    output_dir: str,
    hf_repo: str,
    num_epochs: int = 6,
    max_length: int = 256,
    train_bs: int = 16,
    eval_bs: int = 32, 
):
    train_df = train_df.select([text_col, label_col]).drop_nulls()
    val_df = val_df.select([text_col, label_col]).drop_nulls()
    test_df = test_df.select([text_col, label_col]).drop_nulls()

    mapping = build_label_mapping(train_df[label_col].cast(pl.String).to_list())

    def encode(df: pl.DataFrame) -> pl.DataFrame:
        return df.with_columns(
            pl.col(label_col)
            .cast(pl.String)
            .replace_strict(mapping.label2id)
            .cast(pl.Int64)
            .alias("label_id")
        )

    train_enc, val_enc, test_enc = encode(train_df), encode(val_df), encode(test_df)

    ds = DatasetDict({
        "train": Dataset.from_polars(train_enc.select([text_col, "label_id"])),
        "validation": Dataset.from_polars(val_enc.select([text_col, "label_id"])),
        "test": Dataset.from_polars(test_enc.select([text_col, "label_id"])),
    })

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def preprocess(examples):
        out = tokenizer(examples[text_col], truncation=True, max_length=max_length)
        out["labels"] = examples["label_id"]
        return out

    ds_tok = ds.map(preprocess, batched=True)

    y_train = np.array(train_enc["label_id"].to_list())
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=y_train,
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(mapping.id2label),
        id2label=mapping.id2label,
        label2id=mapping.label2id,
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="best",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_strategy="epoch",
        push_to_hub=True,
        hub_model_id=hf_repo,
        seed=SEED,
        disable_tqdm=True,
        report_to="none",
    )

    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=training_args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["validation"],
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics_factory(num_labels=len(mapping.id2label)),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    train_output = trainer.train()
    val_metrics = trainer.evaluate(ds_tok["validation"], metric_key_prefix="val")
    test_metrics = trainer.evaluate(ds_tok["test"], metric_key_prefix="test")

    return {
        "trainer": trainer,
        "tokenizer": tokenizer,
        "mapping": mapping,
        "train_output": train_output,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }


### A) Conflict Type Classifier (Multiclass)

In [46]:
from huggingface_hub import login, create_repo
from social_conflicts_peru.config import settings

login(settings.HF_TOKEN)

create_repo(
    repo_id="cesarnunezh/distilbert-conflict-classifier",
    private=True,
    exist_ok= True
)

create_repo(
    repo_id="cesarnunezh/berto-conflict-classifier",
    private=True,
    exist_ok= True
)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


RepoUrl('https://huggingface.co/cesarnunezh/berto-conflict-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='cesarnunezh/berto-conflict-classifier')

In [121]:
# Conflict-level dataset: one row per conflict_uid
conflict_df = (
    df_final
    .select(["conflict_uid", "conflict_type", "clean_conflict_text", "location", "date_init", "actors_1", "actors_2", "actors_3"])
    .drop_nulls()
    .unique(subset=["conflict_uid"], keep="first")
    .with_columns(
        pl.format("[CONFLICT] {} [LOCATION] {} [DATE] {} [ACTORS 1] {} [ACTORS 2] {} [ACTORS 3] {}", 
                  pl.col("clean_conflict_text"), pl.col("location"), pl.col("date_init"), pl.col("actors_1"), pl.col("actors_2"), pl.col("actors_3"))
                  .alias("model_text")
    )
)

# Remove very rare labels if needed
min_count = 10
valid_labels = (
    conflict_df["conflict_type"]
    .value_counts()
    .filter(pl.col("count") >= min_count)["conflict_type"]
    .to_list()
)
conflict_df = conflict_df.filter(pl.col("conflict_type").is_in(valid_labels))

train_conflict, temp_conflict = train_test_split(
    conflict_df.to_pandas(),
    test_size=0.30,
    stratify=conflict_df["conflict_type"].to_pandas(),
    random_state=SEED,
)

val_conflict, test_conflict = train_test_split(
    temp_conflict,
    test_size=0.50,
    stratify=temp_conflict["conflict_type"],
    random_state=SEED,
)

train_conflict = pl.from_pandas(train_conflict)
val_conflict = pl.from_pandas(val_conflict)
test_conflict = pl.from_pandas(test_conflict)

print("Conflict split sizes:", train_conflict.height, val_conflict.height, test_conflict.height)
print("Train class counts:")
print(train_conflict["conflict_type"].value_counts().sort("count", descending=True))

Conflict split sizes: 172 37 37
Train class counts:
shape: (5, 2)
┌──────────────────────────────┬───────┐
│ conflict_type                ┆ count │
│ ---                          ┆ ---   │
│ str                          ┆ u32   │
╞══════════════════════════════╪═══════╡
│ socioambiental               ┆ 112   │
│ asuntos de gobierno nacional ┆ 22    │
│ comunal                      ┆ 16    │
│ asuntos de gobierno regional ┆ 12    │
│ asuntos de gobierno local    ┆ 10    │
└──────────────────────────────┴───────┘


In [123]:
CONFLICT_MODEL = "distilbert/distilbert-base-multilingual-cased"

conflict_run = train_text_classifier(
    train_df=train_conflict,
    val_df=val_conflict,
    test_df=test_conflict,
    text_col="model_text",
    label_col="conflict_type",
    model_name=CONFLICT_MODEL,
    output_dir=str(directories.ROOT_DIR / "nlp_analysis/models_conflict_type"),
    hf_repo="cesarnunezh/distilbert-conflict-classifier",
    num_epochs=10,
    max_length=256,
)

print("Validation metrics:", conflict_run["val_metrics"])
print("Test metrics:", conflict_run["test_metrics"])


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1605.94it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '1.621', 'grad_norm': '2.905', 'learning_rate': '1.818e-05', 'epoch': '1'}
{'eval_loss': '1.585', 'eval_accuracy': '0.5946', 'eval_balanced_accuracy': '0.2417', 'eval_precision_weighted': '0.4811', 'eval_recall_weighted': '0.5946', 'eval_f1_weighted': '0.5315', 'eval_f1_macro': '0.2222', 'eval_runtime': '2.909', 'eval_samples_per_second': '12.72', 'eval_steps_per_second': '0.688', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.33it/s]


{'loss': '1.585', 'grad_norm': '3.014', 'learning_rate': '1.618e-05', 'epoch': '2'}
{'eval_loss': '1.547', 'eval_accuracy': '0.6486', 'eval_balanced_accuracy': '0.55', 'eval_precision_weighted': '0.7006', 'eval_recall_weighted': '0.6486', 'eval_f1_weighted': '0.6339', 'eval_f1_macro': '0.4345', 'eval_runtime': '2.871', 'eval_samples_per_second': '12.89', 'eval_steps_per_second': '0.697', 'epoch': '2'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.27it/s]


{'loss': '1.536', 'grad_norm': '3.676', 'learning_rate': '1.418e-05', 'epoch': '3'}
{'eval_loss': '1.508', 'eval_accuracy': '0.7027', 'eval_balanced_accuracy': '0.4417', 'eval_precision_weighted': '0.6164', 'eval_recall_weighted': '0.7027', 'eval_f1_weighted': '0.6218', 'eval_f1_macro': '0.3806', 'eval_runtime': '2.907', 'eval_samples_per_second': '12.73', 'eval_steps_per_second': '0.688', 'epoch': '3'}
{'loss': '1.476', 'grad_norm': '3.792', 'learning_rate': '1.218e-05', 'epoch': '4'}
{'eval_loss': '1.444', 'eval_accuracy': '0.7027', 'eval_balanced_accuracy': '0.4417', 'eval_precision_weighted': '0.6164', 'eval_recall_weighted': '0.7027', 'eval_f1_weighted': '0.6218', 'eval_f1_macro': '0.3806', 'eval_runtime': '2.87', 'eval_samples_per_second': '12.89', 'eval_steps_per_second': '0.697', 'epoch': '4'}
{'train_runtime': '64.74', 'train_samples_per_second': '26.57', 'train_steps_per_second': '1.699', 'train_loss': '1.554', 'epoch': '4'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'val_loss': '1.444', 'val_accuracy': '0.7027', 'val_balanced_accuracy': '0.4417', 'val_precision_weighted': '0.6164', 'val_recall_weighted': '0.7027', 'val_f1_weighted': '0.6218', 'val_f1_macro': '0.3806', 'val_runtime': '2.976', 'val_samples_per_second': '12.43', 'val_steps_per_second': '0.672', 'epoch': '4'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '1.526', 'test_accuracy': '0.7568', 'test_balanced_accuracy': '0.4467', 'test_precision_weighted': '0.738', 'test_recall_weighted': '0.7568', 'test_f1_weighted': '0.7223', 'test_f1_macro': '0.4418', 'test_runtime': '2.907', 'test_samples_per_second': '12.73', 'test_steps_per_second': '0.688', 'epoch': '4'}
Validation metrics: {'val_loss': 1.4442555904388428, 'val_accuracy': 0.7027027027027027, 'val_balanced_accuracy': 0.4416666666666667, 'val_precision_weighted': 0.6163905841325196, 'val_recall_weighted': 0.7027027027027027, 'val_f1_weighted': 0.6217854217854217, 'val_f1_macro': 0.38060606060606056, 'val_runtime': 2.9764, 'val_samples_per_second': 12.431, 'val_steps_per_second': 0.672, 'epoch': 4.0}
Test metrics: {'test_loss': 1.5261814594268799, 'test_accuracy': 0.7567567567567568, 'test_balanced_accuracy': 0.44666666666666666, 'test_precision_weighted': 0.7379665379665379, 'test_recall_weighted': 0.7567567567567568, 'test_f1_weighted': 0.7223047223047222, 'test_f1_macro

In [122]:
# CONFLICT_MODEL = "distilbert/distilbert-base-multilingual-cased"
CONFLICT_MODEL = "dccuchile/bert-base-spanish-wwm-cased"

conflict_run = train_text_classifier(
    train_df=train_conflict,
    val_df=val_conflict,
    test_df=test_conflict,
    text_col="model_text",
    label_col="conflict_type",
    model_name=CONFLICT_MODEL,
    output_dir=str(directories.ROOT_DIR / "nlp_analysis/models_conflict_type"),
    hf_repo="cesarnunezh/berto-conflict-classifier",
    num_epochs=10,
    max_length=256,
)

print("Validation metrics:", conflict_run["val_metrics"])
print("Test metrics:", conflict_run["test_metrics"])


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 19172.96it/s]
BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECT

{'loss': '1.619', 'grad_norm': '6.936', 'learning_rate': '1.818e-05', 'epoch': '1'}
{'eval_loss': '1.547', 'eval_accuracy': '0.5676', 'eval_balanced_accuracy': '0.3583', 'eval_precision_weighted': '0.5952', 'eval_recall_weighted': '0.5676', 'eval_f1_weighted': '0.5622', 'eval_f1_macro': '0.2222', 'eval_runtime': '6.71', 'eval_samples_per_second': '5.514', 'eval_steps_per_second': '0.298', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]


{'loss': '1.568', 'grad_norm': '9.922', 'learning_rate': '1.618e-05', 'epoch': '2'}
{'eval_loss': '1.446', 'eval_accuracy': '0.6757', 'eval_balanced_accuracy': '0.3917', 'eval_precision_weighted': '0.4878', 'eval_recall_weighted': '0.6757', 'eval_f1_weighted': '0.5637', 'eval_f1_macro': '0.2786', 'eval_runtime': '7.375', 'eval_samples_per_second': '5.017', 'eval_steps_per_second': '0.271', 'epoch': '2'}


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]


...

{'loss': '0.89', 'grad_norm': '8.277', 'learning_rate': '1.018e-05', 'epoch': '5'}
{'eval_loss': '1.035', 'eval_accuracy': '0.8108', 'eval_balanced_accuracy': '0.5583', 'eval_precision_weighted': '0.7711', 'eval_recall_weighted': '0.8108', 'eval_f1_weighted': '0.7818', 'eval_f1_macro': '0.5275', 'eval_runtime': '7.334', 'eval_samples_per_second': '5.045', 'eval_steps_per_second': '0.273', 'epoch': '5'}
{'loss': '0.6897', 'grad_norm': '14.47', 'learning_rate': '8.182e-06', 'epoch': '6'}
{'eval_loss': '0.9684', 'eval_accuracy': '0.7838', 'eval_balanced_accuracy': '0.5417', 'eval_precision_weighted': '0.7422', 'eval_recall_weighted': '0.7838', 'eval_f1_weighted': '0.7567', 'eval_f1_macro': '0.5111', 'eval_runtime': '7.327', 'eval_samples_per_second': '5.05', 'eval_steps_per_second': '0.273', 'epoch': '6'}
{'train_runtime': '609.3', 'train_samples_per_second': '2.823', 'train_steps_per_second': '0.181', 'train_loss': '1.216', 'epoch': '6'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'val_loss': '0.9684', 'val_accuracy': '0.7838', 'val_balanced_accuracy': '0.5417', 'val_precision_weighted': '0.7422', 'val_recall_weighted': '0.7838', 'val_f1_weighted': '0.7567', 'val_f1_macro': '0.5111', 'val_runtime': '7.33', 'val_samples_per_second': '5.047', 'val_steps_per_second': '0.273', 'epoch': '6'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '1.034', 'test_accuracy': '0.7838', 'test_balanced_accuracy': '0.5983', 'test_precision_weighted': '0.8041', 'test_recall_weighted': '0.7838', 'test_f1_weighted': '0.7777', 'test_f1_macro': '0.5434', 'test_runtime': '7.338', 'test_samples_per_second': '5.042', 'test_steps_per_second': '0.273', 'epoch': '6'}
Validation metrics: {'val_loss': 0.9684414267539978, 'val_accuracy': 0.7837837837837838, 'val_balanced_accuracy': 0.5416666666666667, 'val_precision_weighted': 0.7421879021879022, 'val_recall_weighted': 0.7837837837837838, 'val_f1_weighted': 0.756706613849471, 'val_f1_macro': 0.5111317254174397, 'val_runtime': 7.3304, 'val_samples_per_second': 5.047, 'val_steps_per_second': 0.273, 'epoch': 6.0}
Test metrics: {'test_loss': 1.0340957641601562, 'test_accuracy': 0.7837837837837838, 'test_balanced_accuracy': 0.5983333333333334, 'test_precision_weighted': 0.8040540540540541, 'test_recall_weighted': 0.7837837837837838, 'test_f1_weighted': 0.7777047564281606, 'test_f1_macro': 

### B) Dialogo vs No Dialogo Classifier (Binary, Group Split by Conflict)

In [49]:
create_repo(
    repo_id="cesarnunezh/distilbert-event-classifier",
    private=True,
    exist_ok= True
)

create_repo(
    repo_id="cesarnunezh/berto-event-classifier",
    private=True,
    exist_ok= True
)

RepoUrl('https://huggingface.co/cesarnunezh/berto-event-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='cesarnunezh/berto-event-classifier')

In [50]:
# Event-level dataset for dialogo classification
dialogo_df = (
    df_final
    .select(["conflict_uid", "dialogo", "clean_event_text", "clean_conflict_text"])
    .drop_nulls()
    .with_columns(
        pl.format("[CONFLICT] {} [EVENT] {}", pl.col("clean_conflict_text"), pl.col("clean_event_text")).alias("model_text")
    )
)

# Group split by conflict_uid to avoid leakage across train/val/test
all_groups = dialogo_df["conflict_uid"].to_pandas()

gss_1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_val_idx, test_idx = next(gss_1.split(dialogo_df.to_pandas(), groups=all_groups))

train_val_df = dialogo_df[train_val_idx]
test_df = dialogo_df[test_idx]

gss_2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=SEED)  # ~15% of total
train_idx, val_idx = next(gss_2.split(train_val_df.to_pandas(), groups=train_val_df["conflict_uid"].to_pandas()))

train_df = train_val_df[train_idx]
val_df = train_val_df[val_idx]

print("Dialogo split sizes:", train_df.height, val_df.height, test_df.height)
print("Train dialogo ratio:")
print(train_df["dialogo"].value_counts().sort("dialogo"))
print("Unique conflicts by split:")
print({
    "train": train_df["conflict_uid"].n_unique(),
    "val": val_df["conflict_uid"].n_unique(),
    "test": test_df["conflict_uid"].n_unique(),
})

Dialogo split sizes: 2112 465 368
Train dialogo ratio:
shape: (2, 2)
┌─────────┬───────┐
│ dialogo ┆ count │
│ ---     ┆ ---   │
│ i8      ┆ u32   │
╞═════════╪═══════╡
│ 0       ┆ 533   │
│ 1       ┆ 1579  │
└─────────┴───────┘
Unique conflicts by split:
{'train': 199, 'val': 43, 'test': 43}


In [51]:
# Keep labels as strings for a stable, explicit mapping
train_df = train_df.with_columns(pl.when(pl.col("dialogo") == 1).then(pl.lit("HAY_DIALOGO")).otherwise(pl.lit("NO_DIALOGO")).alias("dialogo_label"))
val_df = val_df.with_columns(pl.when(pl.col("dialogo") == 1).then(pl.lit("HAY_DIALOGO")).otherwise(pl.lit("NO_DIALOGO")).alias("dialogo_label"))
test_df = test_df.with_columns(pl.when(pl.col("dialogo") == 1).then(pl.lit("HAY_DIALOGO")).otherwise(pl.lit("NO_DIALOGO")).alias("dialogo_label"))

# DIALOGO_MODEL = "distilbert/distilbert-base-multilingual-cased"
DIALOGO_MODEL = "dccuchile/bert-base-spanish-wwm-cased"

dialogo_run = train_text_classifier(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    text_col="model_text",
    label_col="dialogo_label",
    model_name=DIALOGO_MODEL,
    output_dir=str(directories.ROOT_DIR / "nlp_analysis/models_dialogo"),
    hf_repo="cesarnunezh/berto-event-classifier",
    num_epochs=6,
    max_length=256,
)

print("Validation metrics:", dialogo_run["val_metrics"])
print("Test metrics:", dialogo_run["test_metrics"])


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 49492.54it/s]
BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECT

{'loss': '0.564', 'grad_norm': '13.8', 'learning_rate': '1.669e-05', 'epoch': '1'}
{'eval_loss': '0.4771', 'eval_accuracy': '0.7699', 'eval_balanced_accuracy': '0.7885', 'eval_precision_weighted': '0.8186', 'eval_recall_weighted': '0.7699', 'eval_f1_weighted': '0.7804', 'eval_f1_macro': '0.7462', 'eval_pr_auc_positive': '0.6683', 'eval_runtime': '118.6', 'eval_samples_per_second': '3.921', 'eval_steps_per_second': '0.126', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]


{'loss': '0.3912', 'grad_norm': '10.35', 'learning_rate': '1.336e-05', 'epoch': '2'}
{'eval_loss': '0.6437', 'eval_accuracy': '0.7828', 'eval_balanced_accuracy': '0.6774', 'eval_precision_weighted': '0.7695', 'eval_recall_weighted': '0.7828', 'eval_f1_weighted': '0.7669', 'eval_f1_macro': '0.6945', 'eval_pr_auc_positive': '0.6406', 'eval_runtime': '119.2', 'eval_samples_per_second': '3.902', 'eval_steps_per_second': '0.126', 'epoch': '2'}
{'loss': '0.265', 'grad_norm': '25.16', 'learning_rate': '1.003e-05', 'epoch': '3'}
{'eval_loss': '0.6381', 'eval_accuracy': '0.7785', 'eval_balanced_accuracy': '0.7592', 'eval_precision_weighted': '0.7968', 'eval_recall_weighted': '0.7785', 'eval_f1_weighted': '0.7846', 'eval_f1_macro': '0.7415', 'eval_pr_auc_positive': '0.6089', 'eval_runtime': '118.5', 'eval_samples_per_second': '3.923', 'eval_steps_per_second': '0.127', 'epoch': '3'}
{'train_runtime': '1552', 'train_samples_per_second': '8.165', 'train_steps_per_second': '0.51', 'train_loss': '0.4

early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'val_loss': '0.6381', 'val_accuracy': '0.7785', 'val_balanced_accuracy': '0.7592', 'val_precision_weighted': '0.7968', 'val_recall_weighted': '0.7785', 'val_f1_weighted': '0.7846', 'val_f1_macro': '0.7415', 'val_pr_auc_positive': '0.6089', 'val_runtime': '119.5', 'val_samples_per_second': '3.89', 'val_steps_per_second': '0.125', 'epoch': '3'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '0.8295', 'test_accuracy': '0.7663', 'test_balanced_accuracy': '0.7205', 'test_precision_weighted': '0.7762', 'test_recall_weighted': '0.7663', 'test_f1_weighted': '0.7704', 'test_f1_macro': '0.7118', 'test_pr_auc_positive': '0.5339', 'test_runtime': '93.64', 'test_samples_per_second': '3.93', 'test_steps_per_second': '0.128', 'epoch': '3'}
Validation metrics: {'val_loss': 0.6381338238716125, 'val_accuracy': 0.7784946236559139, 'val_balanced_accuracy': 0.7591848450057406, 'val_precision_weighted': 0.7968412901104176, 'val_recall_weighted': 0.7784946236559139, 'val_f1_weighted': 0.7845975631636048, 'val_f1_macro': 0.7414560942299284, 'val_pr_auc_positive': 0.6088914886755934, 'val_runtime': 119.5423, 'val_samples_per_second': 3.89, 'val_steps_per_second': 0.125, 'epoch': 3.0}
Test metrics: {'test_loss': 0.8295084238052368, 'test_accuracy': 0.7663043478260869, 'test_balanced_accuracy': 0.7204837490551776, 'test_precision_weighted': 0.7761535986763488, 'test_recall_weighted'

### Inference Example (Correct Pipeline Task)

In [52]:
from transformers import pipeline

DISTIL_MODEL = "distilbert/distilbert-base-multilingual-cased"
BERTO_MODEL = "dccuchile/bert-base-spanish-wwm-cased"

distilbert_conflict_classifier = pipeline(
    "text-classification",
    model="cesarnunezh/distilbert-conflict-classifier",
    tokenizer=AutoTokenizer.from_pretrained(DISTIL_MODEL),
    truncation=True,
    max_length=256,
)

distilbert_dialogo_classifier = pipeline(
    "text-classification",
    model="cesarnunezh/distilbert-event-classifier",
    tokenizer= AutoTokenizer.from_pretrained(DISTIL_MODEL),
    truncation=True,
    max_length=256,
)

berto_conflict_classifier = pipeline(
    "text-classification",
    model="cesarnunezh/berto-conflict-classifier",
    tokenizer=AutoTokenizer.from_pretrained(BERTO_MODEL),
    truncation=True,
    max_length=256,
)

berto_dialogo_classifier = pipeline(
    "text-classification",
    model="cesarnunezh/berto-event-classifier",
    tokenizer= AutoTokenizer.from_pretrained(BERTO_MODEL),
    truncation=True,
    max_length=256,
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3645.31it/s]


In [124]:
# Sample cases from 2018
# https://www.defensoria.gob.pe/wp-content/uploads/2019/01/Conflictos-Sociales-N%C2%B0-178-Diciembre-2018.pdf

sample_conflicts = [
    {'label' : "HAY_DIALOGO",
     'conflict_type'  : "asuntos de gobierno regional",
     'date_init' : """Diciembre de 2018.""",
     'location' : """Distrito de Huaraz, provincia de Huaraz, región
Áncash.""",
     'actors_1' : 'CISEA Nicrupamapa, I.E.I. Nicrupampa.',
     'actors_2' : """Dirección Regional de Salud de Ancash
(DIRESA), Dirección Regional de Educación.""",
     'actors_3' : """Defensoría del Pueblo, Secretaría de Gestión
Social y Diálogo de la Presidencia del Consejo de Ministros.""",
     'conflict_text' : """El Centro Integrado de Salud, Educación y Agricultura
(CISEA) de Nicrupampa pretende recuperar un terreno de 200
metros para la construcción de un centro materno infantil, que
está actualmente ocupado por la Institución Educativa Inicial de
Nicrupampa""",
     'event_text' : """El 18 de diciembre, se realizó una reunión en la Institución
Educativa con representantes del Dirección Regional de Salud y
El 18 de diciembre, se reunieron en la Institución Educativa
Inicial de Nicrupampa (I.E.I.) representantes de la Procuraduría
del Gobierno Regional de Áncash, el Director de la Red de Salud
de Huaylas Sur y la DIrectora de la Institución Educativa para
tratar el problema del terreno en cuestión. El representante de la
pPocuraduría del Gobierno Regional de Áncash señaló que
podrían realizar un desalojo administrativo. Por su parte, la
directora de la Institución Educativa mencionó que tenía un acta
de donación otorgada por el ex gobernador regional de Áncash.
El 19 de diciembre, mediante el Oficio N°1313-2018-DP/ODANC, la Defensoría del Pueblo hizo traslado al Director de la
Red de Salud de Huaylas Sur de la queja interpuesta por una
ciudadana contra el Centro de Salud de Nicrupampa. Personal
de este centro de salud le informó que no iba atender al público
porque saldrían a realizar una movilización. Asimismo, señaló
que tampoco atendían a las personas que estaban en la cola
desde temprano.
El 3 de enero, la Defensoría se reunió con el Jefe de la Unidad
del Personal del Centro de Salud de Nicrupampa, quien
manifestó que no existe una disposición que autorice la
suspensión de las labores del personal de salud y que
procederán a la constatación de la atención al público. Además,
se comprometió a restablecer la atención a los ciudadanos.
Asimismo, el 7 de enero el Jefe de la Unidad de Personal y al
Asesor Legal de la Red de Salud de Nicrupampa informaron a la
Defensoría del Pueblo que agilizarían los trámites para retomar
las actividades del Centro de Salud. 
El 8 de enero, la Defensoría del Pueblo envió al Director
Ejecutivo de Red de Salud el Oficio N°005-2019-DP/OD-ANC,
donde se le recomienda disponer lo siguiente:
- Adoptar medidas correcticas para garantizar la atención de
los pacientes del Centro de Salud Nicrupampa.
- Emitir un pronunciamiento sobre la legalidad de la medida
de protesta adoptada por el Centro de Salud e implementar
las acciones administrativas disciplinarias.
- Adoptar acciones a fin que este tipo de situación no vuelvan
a repetirse y coordinar para optimizar los flujos de atención a los pacientes.
El mismo día, la Defensoría se reunió con la Gerencia de
Desarrollo Social del Gobierno Regional de Áncash para abordar
el problema entre el CISEA de Nicrupampa y la I.E.I. En esta
reunión, también participaron representantes de la Secretaria de
Gestión Social y Diálogo (SGSD) de la PCM, el Ministerio de
Agricultura y Riego y la Directora de la Institución Educativa. Se
acordó una próxima reunión el 15 de enero para abordar el tema
con presencia de todos los interesados y directivos tanto del
sector salud como de educación de la región."""},
    {'label': 'NO_DIALOGO',
     'conflict_type' : 'socioambiental',
     'date_init' : """Octubre de 2018.""",
     'location' : """Distrito de Quillo, provincia de Yungay, región
Áncash.""",
     'actors_1' : """Comunidad campesina Vìrgen del Rosario de
Quillo y empresa COPEMINA. """,
     'actors_2' : """Dirección Regional de Energía y Minas del
Gobierno Regional de Áncash, Dirección Regional de Salud del
Gobierno Regional de Áncash, Fiscalía Especializada en Medio
Ambiente de Huaraz, Centro de Salud del Centro Poblado de
Huacho, Red de Salud Pacífico Sur, Policía Nacional del Perú,
Congregación religiosa Hermanas del Buen Socorro en el Perú.
""",
     'actors_3' : """Defensoría del Pueblo.""",
     'conflict_text' : """La Comunidad Campesina Virgen del Rosario de Quillo
demanda la intervención de la Dirección Regional de Energía y
Minas, Dirección Regional de Salud y de la Autoridad Nacional
del Agua, debido a una presunta afectación a la salud por las
actividades mineras de la empresa COPEMINA, cuyo
campamento minero se encuentra en la parte alta de la cuenca
Sechín, próximo a la fuente de agua que abastece al centro
poblado. Asimismo, denuncian que en el cerro Huancapampa
se estaría realizando minería informal, sin fiscalización de las
autoridades respectivas.""",
     'event_text': """El 9 de enero, la Defensoría del Pueblo se reunió con la
Dirección Regional de Salud del Gobierno Regional, con el
objetivo de supervisar si se procedió a realizar la evaluación
médica de la de los pobladores de la comunidad Virgen del
Rosario, en el distrito de Quillo, tras la denuncia de
contaminación con plomo en la sangre. Se tomó conocimiento
que no se realizaron las evaluaciones, motivo por el cual se
exhortó a priorizar estas acciones a fin de prevenir incurrir en
delitos por la omisión o demoras de actos funcionales.
Cabe informar que en atención a esta problemática, la
Defensoría del Pueblo cursó el Oficio N° 0704-2018-DP/ODANC a la Dirección Regional de Energía y Minas de Ancash. La
referida institución, mediante Oficio N° 1854-2018-GRA/DREM,
dio respuesta que el Consorcio Peruano de Minas S.A.C. expuso
que el 13 de agosto de 2018, ha sufrido la paralización forzada
de sus operaciones en la Concesión Minera El Extraño, por un
periodo indeterminado y que no tienen acceso a las
instalaciones de las operaciones. Asimismo, con Oficio N° 1940-
2018-GRA/DREM, informó con relación a los permisos con los
que cuenta el Consorcio Peruano de Minas S.A.C.
1. Cuenta con la aprobación de la Declaración de Impacto
Ambiental (DIA) del proyecto de explotación minera
U.E.A.COPEMINA, en la Unidad Económica
Administrativa COPEMINA.
2. Cuenta con la autorización de inicio de actividades de
desarrollo, preparación del proyecto explotación en la
UEA COPEMINA en la concesión minera El Extraño.
3. Cuenta con certificado de operación minera periodo 2018.
4. Actualmente tiene una paralización forzada de sus
operaciones en la concesión minera El Extraño.
Por su parte, la Autoridad Nacional del Agua, con Oficio N° 408-
2018-ANA-AAA.HCH-ALACHUARMEY, en atención al pedido
realizado por la Defensoría del Pueblo, con Oficio N° 0804-2018-
DP/OD-ANC, informó lo siguiente:
- Al Centro Poblado menor de Huacho no se le entregó
licencia de uso de agua con fines poblacionales, ni se
cuenta con trámites en curso solicitados por la JASS y/o
Municipalidad.
- A la Empresa Peruana de Minas S.A.C. COPEMINA, no
se le entregó licencia de uso de agua con fines mineros,
estando a la fecha sin iniciar algún trámite y/o solicitud.
La DIRESA Ancash, con Oficio N° 002427-2018-Region Ancash
– DIRES- DESI/DAISCS/PP ENT – M.P. hizo presente el informe
sobre la condición de salud, anemia y desnutrición de los niños y
niñas y población en general del Centro Poblado de Huacho –
Quillo – Yungay. Se precisó que la Red Pacífico Sur se
encuentra levantando información en el campo (Huacho) y está
realizando el muestreo de agua para los análisis
correspondientes.
El 25 de setiembre, la Red de Salud Pacífico Sur, mediante
Oficio N°1706-2018-GRA/DIRESA/RSPS/ODI/USC/ASA/PVICA, alcanzó a la Defensoría del Pueblo los resultados de la
situación actual de la localidad de Huacho, distrito de Quillo,
Provincia de Yungay (Informe Técnico N° 077-
GRA/DIRESA/RSP-S/ODI/USC/ASA/PVICA/LLR), el cual se
arriban a las siguientes conclusiones:
- El agua de la localidad de Huacho cuenta con un sistema
de cloración ( Clorinador automático),
- Hay presencia de cloro pero en baja concentración,
- Se recomienda reactivar el sistema de cloración lo más
pronto posible,
Deberá realizar monitoreo mensual para verificar la carga
microbiana del sector con los análisis microbiológicos que serán
tomados en la localidad."""},
    {'label': 'HAY_DIALOGO',
     'conflict_type' : 'socioambiental',
     'date_init' : """Noviembre de 2011.""",
     'location' : """Provincias de Huari y Recuay, región Áncash.""",
     'actors_1' : """Asociación de Municipalidades de Centros
Poblados (AMUCEPS) de Huari, Compañía Minera Antamina
S.A. (CMA), Nyrstar, comunidad campesina Cátac, Federación
Agraria Departamental de Áncash (FADA).""",
     'actors_2' : """Ministerio de Energía y Minas (MINEM),
Ministerio del Ambiente (MINAM), Ministerio de Inclusión social
(MIDIS), Ministerio de Economía (MEF), Dirección General de
Infraestructura Agraria y Riego (DGIAR) del Ministerio de
Agricultura y Riego (MINAGRI), Programa de Desarrollo
Productivo Agrario Rural (AGRORURAL), Programa
Subsectorial de Irrigaciones (PSI), Ministerio de Salud (MINSA),
Sub Región Conchucos, Municipalidades de Huari, Chavín de
Huántar y San Marcos.""",
     'actors_3' : """Oficina General de Gestión Social del
Ministerio de Energía y Minas (OGGS), Defensoría del Pueblo,
Obispado de Huari, Comisión Episcopal de Acción Social
(CEAS), Secretaria de Gestión Social y Diálogo de la
Presidencia del Consejo de Ministros (SGSD-PCM).
""",
     'conflict_text' : """La Asociación de Municipalidades de Centros Poblados
(AMUCEPS) de Huari en la provincia de Huari denuncia el
incumplimiento de las empresas mineras Antamina S.A. y
Nyrstar de sus compromisos de responsabilidad social y por los
impactos generados en el medio ambiente.""",
     'event_text': """El 12 de diciembre, la Defensoría del Pueblo, sostuvo una
reunión con la SGSD - PCM, MINEM, Compañía Minera
Antamina S.A., con el objetivo de realizar el seguimiento al
caso. Se dio cuenta que el 31 de octubre se realizó una reunión
entre AMUCEPS y Antamina, en la cual AMUCEPS solicitó que
el espacio de diálogo se amplíe a una mesa de desarrollo que
cuente con la participación de gobiernos locales ( provincial y
distrital) para atender proyectos de desarrollo y promover
vigilancia ciudadana; que la empresa informe los detalles de la
ejecución del proyecto de forestación denominado Huari I, y se
gestione una reunión con la nueva Directora Ejecutiva de
AGRORURAL, a fin de exponer los alcances del proyecto de
forestación Huari II.
La empresa informó que la consultoría para la adecuación de
los Centros Poblados tiene los Términos de Referencia para ser
puestos a convocatoria. Ambas partes solicitan que se
convoque a una reunión del espacio de diálogo que cuente con
la participación del MINAGRI, PSI, AGRORURAL Y DGIAR,
para que informen el estado de los proyectos de riego. De igual
manera, se ha visto la necesidad de convocar a la Minera Los
Quenuales para que informen el estado de los dos proyectos de
riego a su cargo.
"""},
    {'label': 'HAY_DIALOGO',
     'conflict_type' : 'comunal',
     'date_init' :"""Abril de 2014.""",
     'location' : """Comunidad campesina Lambrama en el distrito de
Lambrama, provincia de Abancay y comunidad campesina
Curpahuasi en el distrito de Curpahuasi, provincia de Grau,
región Apurímac.
""",
     'actors_1' : """Comunidades de Lambrama y Curpahuasi,
alcaldes de los distritos de Lambrama y Curpahuasi.""",
     'actors_2' : """Gerencia Regional del Gobierno Regional
de Apurímac (GRA), Dirección de Demarcación Territorial del
GRA, Sub Gerencia de Saneamiento Físico Legal de la
Propiedad Rural del GRA, Dirección Regional de Agricultura.""",
     'actors_3' : """Defensoría del Pueblo""",
     'conflict_text' : """ Las comunidades campesinas Lambrama y Curpahuasi
se encuentran en disputa por linderos territoriales. Ambas
insisten en que el sector de Taccata pertenece a su jurisdicción.""",
     'event_text': """El 13 de diciembre, se realizó una reunión en las instalaciones
de la OD de Apurímac. En esta reunión fueron convocados los
representantes de las comunidades de Curpahuasi y Lambrama,
no obstante los integrantes de esta última no asistieron.
Asimismo, se contó con la presencia de representantes del
GORE Apurímac, la Región Policial de Apurímac, FORPRAR y
de la OD de Apurímac.
Finalmente, se acordó que FORPRAP convocará a una reunión
para dar conocer el proceso de titulación que se realizará en la
comunidad de Curpahuasi. En esta reunión, se invitará a las
comunidades colindantes."""},
    {'label': 'NO_DIALOGO',
     'conflict_type' : 'socioambiental',
     'date_init' : """Octubre de 2013.""",
     'location' : """Distritos de Deán Valdivia, Cocachacra y Punta de
Bombón, provincia de Islay, región Arequipa.""",
     'actors_1' : """Autoridades (alcaldes de Islay, Cocachacra,
Punta Bombón y Deán Valdivia), agricultores y pobladores de
los distritos de Cocachacra, Deán Valdivia y Punta Bombón de
la provincia de Islay, Frente de Defensa del Valle de Tambo,
Junta de Usuarios Irrigación Ensenada-Mejía-Mollendo, Junta
de Usuarios del Valle de Tambo, empresa minera Southern
Perú Copper Corporation (SPCC).""",
     'actors_2' : """Pobladores de otros distritos de la
provincia de Islay, Federación Departamental de Trabajadores
de Arequipa (FDTA), Frentes de Defensa Macro Regional,
Partido Político Tierra y Libertad, Ministerio de Agricultura y
Riego (MINAGRI), Autoridad Nacional del Agua (ANA),
Ministerio de Energía y Minas (MINEM) y Ministerio del
Ambiente (MINAM), Organismo de Evaluación y Fiscalización
Ambiental (OEFA), Ministerio del Interior (MININTER) - Policía
Nacional de la Policía (PNP), Fuerzas Armadas (FFAA),
Contraloría General de la República, Poder Judicial, Ministerio
Público.""",
     'actors_3' : """ Secretaría de Gestión Social y Diálogo de la
Presidencia del Consejo de Ministros (SGSD-PCM), Gobierno
Regional de Arequipa (GORE Arequipa), Defensoría del Pueblo.""",
     'conflict_text' : """Agricultores, pobladores y autoridades locales de la
provincia de Islay se oponen al proyecto minero Tía María de la
empresa minera Southern Perú Copper Corporation (SPCC) por
el temor de que se generen impactos negativos al ambiente, y
en consecuencia, se afecte la actividad agrícola en la provincia.
Este caso fue reportado en agosto del 2009 hasta abril de 2011
en que se llega a una solución con la emisión de la Resolución
Directoral N.° 105-2011–MEM-AAM del Ministerio de Energía y
Minas que declara inadmisible el Estudio de Impacto Ambiental
del proyecto minero Tía María presentado por la empresa
minera SPCC.""",
     'event_text': """El 3 de diciembre, representantes de la empresa Southern
realizaron una presentación en la ciudad de Arequipa sobre el
proyecto Tía María, señalando que según la encuesta que se
encargó a Ipsos el 59% de la población de la provincia de Islay
está a favor del desarrollo del proyecto, mientras que el 38%
está en contra."""},
    {'label': 'HAY_DIALOGO',
     'conflict_type' : 'asuntos de gobierno nacional',
     'date_init' : """Mayo de 2017.""",
     'location' : """Provincia de Caylloma, región Arequipa""",
     'actors_1' : """Frente de Defensa de la provincia de
Caylloma, Gobierno Regional de Arequipa, Ministerio de
Energía y Minas, Ministerio del Ambiente, Ministerio de
Transportes y Comunicaciones, Ministerio de Agricultura.
Autoridad Autónoma de Majes (AUTODEMA) .""",
     'actors_2' : """""",
     'actors_3' : """Secretaría de Gestión Social y Diálogo de la
Presidencia del Consejo de Ministros (SGSD-PCM), Defensoría
del Pueblo.""",
     'conflict_text' : """Ciudadanos de Caylloma reclaman al Poder Ejecutivo y
al Gobierno Regional de Arequipa tratar sobre la ejecución del
proyecto Majes Siguas II, la represa de Angostura, el asfaltado
de la vía Vizcachani a Orcopampa y la conformación de un
fondo minero""",
     'event_text': """La reunión programada para el 14 de diciembre, de acuerdo al
Acta de la sesión de las submesas de trabajo “Carretera
Vizcachani - Caylloma y Majes Siguas II” (suscrita el 21.11.2018,
en la sede de la Alcadía Provincial de Caylloma) fue
reprogramada para que se desarrolle con las nuevas
autoridades regional y municipales."""}               
]

#### Model evaluation

##### Conflict type classifier

In [127]:
final_count: dict[str,int] = {}
for conflict in sample_conflicts:

    conflict_type = conflict['conflict_type']
    conflict_text = conflict['conflict_text']
    location = conflict['location']
    date_init = conflict['date_init']
    actors_1 = conflict['actors_1']
    actors_2 = conflict['actors_2']
    actors_3 = conflict['actors_3']

    mode_text = f"""[CONFLICT] {conflict_text} [LOCATION] {location} [DATE] {date_init} [ACTORS 1] {actors_1} [ACTORS 2] {actors_2} [ACTORS 3] {actors_3}"""
    distilbert_result = distilbert_conflict_classifier(mode_text)
    berto_result = berto_conflict_classifier(mode_text)

    print(f"True label: {conflict_type} | Distilbert: {distilbert_result[0]['label']}, Berto: {berto_result[0]['label']}")
    if distilbert_result[0]['label'] == conflict_type:
        final_count['distilbert'] = final_count.get('distilbert', 0) + 1
    if berto_result[0]['label'] == conflict_type:
        final_count['berto'] = final_count.get('berto', 0) + 1

print(f"""
Final results: 
    Distilbert: {final_count.get('distilbert', 0)}/{len(sample_conflicts)}
    Berto: {final_count.get('berto', 0)}/{len(sample_conflicts)}
""")

True label: asuntos de gobierno regional | Distilbert: asuntos de gobierno regional, Berto: asuntos de gobierno local
True label: socioambiental | Distilbert: comunal, Berto: socioambiental
True label: socioambiental | Distilbert: asuntos de gobierno nacional, Berto: socioambiental
True label: comunal | Distilbert: comunal, Berto: asuntos de gobierno local
True label: socioambiental | Distilbert: asuntos de gobierno regional, Berto: socioambiental
True label: asuntos de gobierno nacional | Distilbert: asuntos de gobierno regional, Berto: socioambiental

Final results: 
    Distilbert: 2/6
    Berto: 3/6



In [126]:
final_count: dict[str,int] = {}
for conflict in sample_conflicts:

    dialogo = conflict['label']
    conflict_text = conflict['conflict_text']
    event_text = conflict['event_text']

    distilbert_result = distilbert_dialogo_classifier(f"[CONFLICT] {conflict_text} [EVENT] {event_text}")
    berto_result = berto_dialogo_classifier(f"[CONFLICT] {conflict_text} [EVENT] {event_text}")

    print(f"True label: {dialogo} | Distilbert: {distilbert_result[0]['label']}, Berto: {berto_result[0]['label']}")
    if distilbert_result[0]['label'] == dialogo:
        final_count['distilbert'] = final_count.get('distilbert', 0) + 1
    if berto_result[0]['label'] == dialogo:
        final_count['berto'] = final_count.get('berto', 0) + 1

print(f"""
Final results: 
    Distilbert: {final_count.get('distilbert', 0)}/{len(sample_conflicts)}
    Berto: {final_count.get('berto', 0)}/{len(sample_conflicts)}
""")

True label: HAY_DIALOGO | Distilbert: NO_DIALOGO, Berto: NO_DIALOGO
True label: NO_DIALOGO | Distilbert: NO_DIALOGO, Berto: NO_DIALOGO
True label: HAY_DIALOGO | Distilbert: HAY_DIALOGO, Berto: HAY_DIALOGO
True label: HAY_DIALOGO | Distilbert: NO_DIALOGO, Berto: NO_DIALOGO
True label: NO_DIALOGO | Distilbert: NO_DIALOGO, Berto: NO_DIALOGO
True label: HAY_DIALOGO | Distilbert: NO_DIALOGO, Berto: NO_DIALOGO

Final results: 
    Distilbert: 3/6
    Berto: 3/6



# Find keywords - events that drive activation of conflicts

# Prompt engineering: which questions works better to find out why does the conflict activates/deactivates

# Agents to biased for active/inactive to predict.